In [1]:
import numpy as np
import json

from posture import gen_posture
from lib import inverse_kinematics
from motion import gen_walk_path, gen_fastwalk_path, gen_turn_path, gen_climb_path
from motion import gen_rotatex_path, gen_rotatey_path, gen_rotatez_path, gen_twist_path

servo_min = 125
servo_max = 575
servo_range = servo_max - servo_min

In [2]:
with open("config.json", "r", encoding="utf-8") as read_file:
    config = json.load(read_file)

In [3]:
standby = gen_posture(60, 75, config)
angles = inverse_kinematics(standby, config)

np.round(angles / 180 * servo_range + servo_min)

array([[350., 425., 388.],
       [350., 275., 312.],
       [350., 275., 312.],
       [350., 275., 312.],
       [350., 425., 388.],
       [350., 425., 388.]])

In [4]:
standby

array([[ 1.06755802e+02,  1.32755802e+02, -5.98377694e+01],
       [ 1.46253482e+02,  0.00000000e+00, -5.98377694e+01],
       [ 1.06755802e+02, -1.32755802e+02, -5.98377694e+01],
       [-1.06755802e+02,  1.32755802e+02, -5.98377694e+01],
       [-1.46253482e+02, -1.33956131e-14, -5.98377694e+01],
       [-1.06755802e+02, -1.32755802e+02, -5.98377694e+01]])

In [36]:
from lib import path_rotate_z

laydown = gen_posture(20, 35, config)
print(laydown)
standing_up_lut_size = 28

lift_up_size = 10
adjust_leg_size = int((standing_up_lut_size - lift_up_size) / 2)

lift_up_linspace = np.linspace(laydown[0, 2], standby[0, 2], lift_up_size)

lut_standup = np.zeros((standing_up_lut_size, 6, 3))

for idx, var in enumerate(lift_up_linspace):
    laydown[:, 2] = var
    lut_standup[idx, :, :] = laydown

radius = (lut_standup[lift_up_size - 1, 1, 0] - standby[1, 0]) / 2

step_angle = np.pi / adjust_leg_size
angle = np.arange(1, adjust_leg_size + 1) * step_angle

z_offset = radius * np.sin(angle)
x_offset = radius * np.cos(angle)- radius

r_leg2_offset = np.zeros((adjust_leg_size, 3))
r_leg2_offset[:, 0] = x_offset
r_leg2_offset[:, 2] = z_offset

r_leg1_offset = path_rotate_z(r_leg2_offset, 45)
r_leg3_offset = path_rotate_z(r_leg2_offset, -45)

l_leg1_offset = path_rotate_z(r_leg2_offset, 135)
l_leg2_offset = path_rotate_z(r_leg2_offset, 180)
l_leg3_offset = path_rotate_z(r_leg2_offset, -135)

for idx in range(0, adjust_leg_size):
    lut_standup[idx+lift_up_size, :, :] = lut_standup[lift_up_size-1, :, :]
    lut_standup[idx+lift_up_size, 3, :] = lut_standup[idx+lift_up_size, 3, :]+l_leg1_offset[idx, :]
    lut_standup[idx+lift_up_size, 1, :] = lut_standup[idx+lift_up_size, 1, :]+r_leg2_offset[idx, :]
    lut_standup[idx+lift_up_size, 5, :] = lut_standup[idx+lift_up_size, 5, :]+l_leg3_offset[idx, :]


for idx in range(0, adjust_leg_size):
    lut_standup[idx+lift_up_size+adjust_leg_size, :, :] = lut_standup[lift_up_size+adjust_leg_size-1, :, :]

    lut_standup[idx+lift_up_size+adjust_leg_size, 0, :] = lut_standup[idx+lift_up_size+adjust_leg_size, 0, :]+r_leg1_offset[idx, :]
    lut_standup[idx+lift_up_size+adjust_leg_size, 4, :] = lut_standup[idx+lift_up_size+adjust_leg_size, 4, :]+l_leg2_offset[idx, :]
    lut_standup[idx+lift_up_size+adjust_leg_size, 2, :] = lut_standup[idx+lift_up_size+adjust_leg_size, 2, :]+r_leg3_offset[idx, :]


[[ 1.24253405e+02  1.50253405e+02 -8.14951501e+00]
 [ 1.70998830e+02  0.00000000e+00 -8.14951501e+00]
 [ 1.24253405e+02 -1.50253405e+02 -8.14951501e+00]
 [-1.24253405e+02  1.50253405e+02 -8.14951501e+00]
 [-1.70998830e+02 -1.64260442e-14 -8.14951501e+00]
 [-1.24253405e+02 -1.50253405e+02 -8.14951501e+00]]


In [38]:
lut_walk_0 = gen_walk_path(standby, direction=0)
lut_walk_180 = gen_walk_path(standby, direction=180)

lut_walk_r45 = gen_walk_path(standby, direction=315)
lut_walk_r90 = gen_walk_path(standby, direction=270)
lut_walk_r135 = gen_walk_path(standby, direction=225)

lut_walk_l45 = gen_walk_path(standby, direction=45)
lut_walk_l90 = gen_walk_path(standby, direction=90)
lut_walk_l135 = gen_walk_path(standby, direction=135)

lut_fast_forward = gen_fastwalk_path(standby, g_steps=28)
lut_fast_backward = gen_fastwalk_path(standby, g_steps=28, reverse=True)

lut_turn_left = gen_turn_path(standby, direction="left")
lut_turn_right = gen_turn_path(standby, direction="right")

lut_climb_forward = gen_climb_path(standby, reverse=False)
lut_climb_backward = gen_climb_path(standby, reverse=True)

lut_rotate_x = gen_rotatex_path(standby, g_steps=28, swing_angle=10, y_radius=10)
lut_rotate_y = gen_rotatey_path(standby, g_steps=28, swing_angle=10, x_radius=10)
lut_rotate_z = gen_rotatez_path(standby, g_steps=28, z_lift=7)

lut_twist = gen_twist_path(standby, g_steps=28)

In [39]:
var_name_list = [
    "lut_walk_0",
    "lut_walk_180",
    "lut_walk_r45",
    "lut_walk_r90",
    "lut_walk_r135",
    "lut_walk_l45",
    "lut_walk_l90",
    "lut_walk_l135",
    "lut_fast_forward",
    "lut_fast_backward",
    "lut_turn_left",
    "lut_turn_right",
    "lut_climb_forward",
    "lut_climb_backward",
    "lut_rotate_x",
    "lut_rotate_y",
    "lut_rotate_z",
    "lut_twist",
    "lut_standup",
]

var_data_list = [
    lut_walk_0,
    lut_walk_180,
    lut_walk_r45,
    lut_walk_r90,
    lut_walk_r135,
    lut_walk_l45,
    lut_walk_l90,
    lut_walk_l135,
    lut_fast_forward,
    lut_fast_backward,
    lut_turn_left,
    lut_turn_right,
    lut_climb_forward,
    lut_climb_backward,
    lut_rotate_x,
    lut_rotate_y,
    lut_rotate_z,
    lut_twist,
    lut_standup,
]

In [40]:
fp = open("./motion.h", "w")

fp.write("/*\n" + " * This is an automatically generated code\n" + " *\n")

fp.write(" */\n\n")

fp.write("#ifndef MOTION_H\n")
fp.write("#define MOTION_H\n\n")

for var_idx, var_name in enumerate(var_name_list):
    var = var_data_list[var_idx]

    fp.write("static int " + var_name + "_length = " + str(np.shape(var)[0]) + ";\n")

    for idx in range(0, np.shape(var)[0]):
        angles = inverse_kinematics(var[idx, :, :], config)
        path_walk_pwm = np.round(angles / 180 * servo_range + servo_min)
        path_walk_pwm = path_walk_pwm.astype(int)

        if idx == 0:
            fp.write(
                "static int " + var_name + "[" + str(np.shape(var)[0]) + "][6][3] = {{"
            )
        else:
            fp.write("                                  {")

        for r in range(0, 6):
            if r > 0:
                fp.write("                                   ")

            fp.write(
                "{"
                + str(path_walk_pwm[r, 0])
                + ", "
                + str(path_walk_pwm[r, 1])
                + ", "
                + str(path_walk_pwm[r, 2])
                + "}"
            )

            if r < 5:
                fp.write(",\n")
            else:
                fp.write("}")
        # fp.write("{"+str(path_walk_pwm[5, 0])+", " +str(path_walk_pwm[5, 1])+", "+str(path_walk_pwm[5, 2])+"}}")

        if idx == (np.shape(var)[0] - 1):
            fp.write("};\n\n")
        else:
            fp.write(",\n")

fp.write("#endif // MOTION_H\n")

fp.close()